# 🔹 1. Імпорти всіх бібліотек

In [ ]:
!pip install langchain_community langchain_text_splitters pyngrok chromadb langchain langchain-core langchain-huggingface langchain-openai sentence-transformers openai "langchain>=0.2.0" langchain-chroma langchain-community "sentence-transformers>=2.2.0" > /dev/null

In [ ]:
!pip install unstructured

In [ ]:
from langchain_openai import ChatOpenAI
print("✅ Успішний імпорт ChatOpenAI!")

✅ Успішний імпорт ChatOpenAI!


In [ ]:
import os
from pathlib import Path
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma as ChromaClient

# 🔹 2. Завантаження текстових файлів

In [ ]:
data_dir = "data/quantum_notes"

txt_loader = DirectoryLoader(
    data_dir,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    recursive=True
)
txt_docs = txt_loader.load()

# додаємо метадані для txt
for d in txt_docs:
    d.metadata["type"] = "txt"

print(f"✅ Завантажено {len(txt_docs)} .txt файлів.")

✅ Завантажено 5 .txt файлів.


In [ ]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

md_loader = DirectoryLoader(
    data_dir,
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader,
    recursive=True
)
md_docs = md_loader.load()

# додаємо метадані для md
for d in md_docs:
    d.metadata["type"] = "md"

print(f"✅ Завантажено {len(md_docs)} .md файлів.")

# Об'єднуємо всі документи
docs = txt_docs + md_docs
print(f"✅ Загальна кількість завантажених файлів: {len(docs)}")

✅ Завантажено 3 .md файлів.
✅ Загальна кількість завантажених файлів: 8


У цьому випадку, коли ви використовуєте `vectorstore.similarity_search(query=query, k=3, filter={"type": "txt"})`, ви запитуєте векторну базу знайти 3 найбільш схожі фрагменти тексту (`k=3`) до вашого запиту (`query`), але **тільки ті фрагменти, у яких метадані містять ключ `type` зі значенням `txt`**.

Це дозволяє обмежувати пошук певними типами документів або джерел, що може бути корисним, якщо у вас є документи різних форматів (наприклад, .txt, .pdf, .docx) або з різних джерел, і ви хочете шукати тільки в певній підмножині даних.

# 🔹 3. Розбиття документів на фрагменти

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
splitted_docs = splitter.split_documents(docs)

print(f"📄 Після розбиття отримано {len(splitted_docs)} фрагментів.")
print("Приклад:\n")
for i, d in enumerate(splitted_docs[:3]):
    print(f"--- Фрагмент {i+1} ---")
    print(d.page_content[:200], "...")
    print("📎 Метадані:", d.metadata, "\n")

📄 Після розбиття отримано 62 фрагментів.
Приклад:

--- Фрагмент 1 ---
Квантова заплутаність — це одне з найдивніших і найконтрінтуїтивніших явищ у фізиці, яке є ключовим ресурсом для квантових обчислень. Це особливий зв'язок, що може виникнути між двома або більше кубіт ...
📎 Метадані: {'source': 'data/quantum_notes/quantum_computer_2.txt', 'type': 'txt'} 

--- Фрагмент 2 ---
система, навіть якщо їх розділити величезними відстанями. Стан одного кубіта стає миттєво залежним від стану іншого. Вимірювання одного з них миттєво визначає результат вимірювання іншого. Альберт Ейн ...
📎 Метадані: {'source': 'data/quantum_notes/quantum_computer_2.txt', 'type': 'txt'} 

--- Фрагмент 3 ---
механіки, назвав це явище "моторошною дією на відстані". Він вважав, що такий миттєвий зв'язок порушує принцип локальності, згідно з яким об'єкт може зазнавати впливу лише свого безпосереднього оточен ...
📎 Метадані: {'source': 'data/quantum_notes/quantum_computer_2.txt', 'type': 'txt'} 



# 🔹 4. Створення та збереження векторної бази Chroma

In [ ]:
persist_dir = "chroma_db"
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=splitted_docs,
    embedding=embedding_model,
    persist_directory=persist_dir
)

print(f"✅ Векторна база створена й збережена в '{persist_dir}'")
print("Кількість збережених векторів:", vectorstore._collection.count())

✅ Векторна база створена й збережена в 'chroma_db'
Кількість збережених векторів: 490


# 🔹 5. Створення retriever’ів (SIMILARITY / MMR)

In [ ]:
retriever_sim = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
retriever_mmr = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 15})

query = "З чим працюють квантвові компютери?"
print(f"\n🔍 Запит: {query}\n")

# --- Режим SIMILARITY ---
print("📘 РЕЖИМ SIMILARITY:")
for i, doc in enumerate(retriever_sim.invoke(query), 1):
    print(f"\n--- Результат {i} ---")
    print(doc.page_content.strip())
    print("📎 Метадані:", doc.metadata)

# --- Режим MMR ---
print("\n📗 РЕЖИМ MMR:")
for i, doc in enumerate(retriever_mmr.invoke(query), 1):
    print(f"\n--- Результат {i} ---")
    print(doc.page_content.strip())
    print("📎 Метадані:", doc.metadata)

# --- Інтерактивна фільтрація ---
print("\n⚙️ ФІЛЬТРАЦІЯ ЗА МЕТАДАНИМИ:")

filter_type = input("Введіть тип фільтрації ('type' для типу файлу, 'source' для конкретного файлу): ").strip().lower()

if filter_type == "type":
    file_type = input("Введіть тип файлу ('md', 'txt', або залиште порожнім для обох): ").strip().lower()
    if file_type in ["md", "txt"]:
        filter_value = file_type
        filter_dict = {"type": filter_value}
        print(f"\n📙 ФІЛЬТР ЗА type={filter_value.upper()}:")
    else:
        filter_dict = {}
        print("\n📙 ФІЛЬТР БЕЗ ОБМЕЖЕНЬ ЗА ТИПОМ:")

    results_filtered = vectorstore.similarity_search(query=query, k=3, filter=filter_dict)

elif filter_type == "source":
    file_source = input("Введіть шлях до файлу (наприклад, data/quantum_notes/quantum_computer_1.txt): ").strip()
    if file_source:
        filter_value = file_source
        filter_dict = {"source": filter_value}
        print(f"\n📙 ФІЛЬТР ЗА source='{filter_value}':")
    else:
        filter_dict = {}
        print("\n📙 ФІЛЬТР БЕЗ ОБМЕЖЕНЬ ЗА ДЖЕРЕЛОМ:")

    results_filtered = vectorstore.similarity_search(query=query, k=3, filter=filter_dict)

else:
    filter_dict = {}
    results_filtered = vectorstore.similarity_search(query=query, k=3)
    print("\n📙 ФІЛЬТР БЕЗ ОБМЕЖЕНЬ:")


for i, doc in enumerate(results_filtered, 1):
    print(f"\n--- Результат {i} ---")
    print(doc.page_content.strip())
    print("📎 Метадані:", doc.metadata)


🔍 Запит: З чим працюють квантвові компютери?

📘 РЕЖИМ SIMILARITY:

--- Результат 1 ---
в основі квантової криптографії, що дозволяє створювати абсолютно захищені канали зв'язку. Будь-яка спроба прослухати такий канал неминуче зруйнує заплутаність і буде негайно виявлена.
📎 Метадані: {'type': 'txt', 'source': 'data/quantum_notes/quantum_computer_2.txt'}

--- Результат 2 ---
в основі квантової криптографії, що дозволяє створювати абсолютно захищені канали зв'язку. Будь-яка спроба прослухати такий канал неминуче зруйнує заплутаність і буде негайно виявлена.
📎 Метадані: {'source': 'data/quantum_notes/quantum_computer_2.txt', 'type': 'txt'}

--- Результат 3 ---
в основі квантової криптографії, що дозволяє створювати абсолютно захищені канали зв'язку. Будь-яка спроба прослухати такий канал неминуче зруйнує заплутаність і буде негайно виявлена.
📎 Метадані: {'type': 'txt', 'source': 'data/quantum_notes/quantum_computer_2.txt'}

--- Результат 4 ---
в основі квантової криптографії, що дозволяє 

# 🔹 6. Підключення локальної LLM через NGROK

In [ ]:
NGROK_URL = "https://aviana-spectatorial-laxly.ngrok-free.dev"
print(f"\n🌍 Підключаємось до тунелю: {NGROK_URL}")
print("🔥 Переконайся, що LM Studio і ngrok запущені!")

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = ChromaClient(embedding_function=embedding_model, persist_directory=persist_dir)
# retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 6})

llm = ChatOpenAI(
    model="local-model",
    base_url=f"{NGROK_URL}/v1",
    api_key="not-needed",
    temperature=0.4,
    max_tokens=1024
)
print("🔗 LLM успішно підключено.")


🌍 Підключаємось до тунелю: https://aviana-spectatorial-laxly.ngrok-free.dev
🔥 Переконайся, що LM Studio і ngrok запущені!
🔗 LLM успішно підключено.


In [ ]:
# Додаємо вибір режиму пошуку
search_mode = input("Оберіть режим пошуку (similarity/mmr): ").strip().lower()

if search_mode == "mmr":
    retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 25})
    print("Використовується режим пошуку MMR.")
else: # За замовчуванням використовуємо similarity
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
    print("Використовується режим пошуку Similarity (за замовчуванням).")

Оберіть режим пошуку (similarity/mmr): mmr
Використовується режим пошуку MMR.


# 🔹 7. Створення RAG-ланцюга

In [ ]:
prompt = ChatPromptTemplate.from_template(
    """Використай наведений контекст, щоб відповісти на запит користувача.
    Якщо в контексті нема відповіді — скажи, що немає достатньо інформації.

    Контекст:
    {context}

    Запит:
    {question}"""
)

rag_chain = (
    {
        "context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 🔹 8. Тестовий запит

In [ ]:
query = "З чим працюють квантвові компютери?"
print("\n⏳ Надсилаю запит до локальної моделі...")
result = rag_chain.invoke(query)

print("\n🧠 Відповідь LLM:\n", result)


⏳ Надсилаю запит до локальної моделі...

🧠 Відповідь LLM:
 <think>
Okay, the user is asking "З чим працюють квантвові компютери?" which translates to "What do quantum computers work with?" 

Looking at the provided context, it mentions that quantum cryptography allows for secure communication channels by leveraging quantum phenomena like entanglement. It also talks about the challenges in maintaining quantum states and how researchers are improving stabilization technologies.

However, the question is about how quantum computers operate, not necessarily about quantum cryptography. The context doesn't explicitly explain the components or mechanisms of quantum computers. It discusses quantum key distribution (QKD) for secure communication, but not the general workings of quantum computers themselves.

The user might be confused between quantum cryptography and quantum computing. The context mentions quantum processors and companies like IBM and Google working on them, but it doesn't det